# risk_budget_residual_drawdown_guard_walkforward_v1

Purpose: validate the fixed residual drawdown guard candidate out-of-sample on the saved `risk_budget_usage_v1` walk-forward actor paths before any runtime promotion.

Governance:
- Canonical baseline: `risk_budget_usage_v1` walk-forward composition of `balanced_rule_residual_mode_v1` + `balanced_rule_residual_imitation_v4`.
- Direct comparison: the same WF composition after the already selected downside-vol and crash-risk overlays.
- Role: `diagnose` / promotion check for `risk_budget_residual_drawdown_guard_v1`.
- Changed layer: `execution` only; no actor retraining.
- External reference: walk-forward / CSCV-style anti-overfit discipline from Bailey et al. PBO work, plus volatility targeting as a standard risk-control mechanism. This notebook keeps the single-run conclusion local to the runtime artifact.
- Acceptance: residual guard should reduce drawdown on the WF chain without losing more than 5pp relative return versus the crash-risk stage; strict target rows (`relative_return >= 30pp` and `max_dd < 15%`) are reported separately because bull windows can legitimately trail buy-and-hold.

Boundary:
Saved WF daily files exist for the slow and fast actors, not for every intermediate execution overlay. This notebook reconstructs the closest verifiable chain: `risk_budget_usage_v1 WF composition -> selected downside-vol target -> selected crash-risk score overlay -> selected residual guard`. Market/score/volatility features use full historical market context, while account drawdown state resets inside each WF window.

In [1]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
import json
import sys

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

pd.set_option("display.max_columns", 140)
pd.set_option("display.width", 220)


def find_repo_root() -> Path:
    start = Path.cwd().resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "AGENTS.md").exists() and (candidate / "runtime").exists():
            return candidate
    raise RuntimeError(f"could not locate repo root from {start}")


ROOT = find_repo_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.soxl_quant.policy.two_speed_budget import TwoSpeedBudgetConfig, compose_budget_usage_policy_frame
EXPERIMENT = "risk_budget_residual_drawdown_guard_walkforward_v1"
CARD = ROOT / "runtime/execution/card" / EXPERIMENT
CARD.mkdir(parents=True, exist_ok=True)
WALK_DIR = CARD / "walkforward"
WALK_DIR.mkdir(parents=True, exist_ok=True)

MODEL_CARD = ROOT / "runtime/model/card"
SLOW_PROFILE = "balanced_rule_residual_mode_v1"
FAST_PROFILE = "balanced_rule_residual_imitation_v4"
BASE_WF_SUMMARY_PATH = MODEL_CARD / "risk_budget_usage_v1/risk_budget_usage_v1_walkforward_composed.json"
PANEL_PATH = ROOT / "data/processed/panels/soxl_soxx_vix_full.csv"
VIX3M_PATH = ROOT / "data/raw/external/VIX3M.csv"
VVIX_PATH = ROOT / "data/raw/external/VVIX.csv"
PREVIOUS_SELECTION_PATH = ROOT / "runtime/execution/card/risk_budget_residual_drawdown_guard_v1/selection.json"

WF_WINDOWS = ["wf_2020", "wf_2022", "wf_2023_2024"]
WF_DATE_RANGES = {
    "wf_2020": ("2020-01-01", "2020-12-31"),
    "wf_2022": ("2022-01-01", "2022-12-31"),
    "wf_2023_2024": ("2023-01-01", "2024-12-31"),
}

DOWNSIDE_CONFIG = {
    "label": "vol_w10_tv0.35_mx4.0_b0.02_trend1",
    "window": 10,
    "target_vol": 0.35,
    "max_scale": 4.0,
    "band": 0.02,
    "trend_filter": True,
}
CRASH_CONFIG = {
    "score_name": "panic_rebound_guarded",
    "entry": 0.70,
    "exit": 0.65,
    "cap": 0.05,
    "min_account_dd": 0.07,
    "trade_band": 0.04,
    "release_mom": False,
}
RESIDUAL_CONFIG = {
    "start_drawdown": 0.09,
    "full_drawdown": 0.22,
    "min_cap": 0.095,
    "trade_band": 0.01,
    "release_drawdown": 0.0,
}


def load_market_panel() -> pd.DataFrame:
    panel = pd.read_csv(PANEL_PATH)
    panel["datetime"] = pd.to_datetime(panel["date"])
    panel = panel.sort_values("datetime").rename(columns={"soxl_close": "panel_soxl_close", "soxx_close": "panel_soxx_close"})
    if VIX3M_PATH.exists():
        vix3m = pd.read_csv(VIX3M_PATH)
        vix3m["datetime"] = pd.to_datetime(vix3m["DATE"], errors="coerce")
        panel = panel.merge(vix3m.rename(columns={"CLOSE": "vix3m"})[["datetime", "vix3m"]], on="datetime", how="left")
    else:
        panel["vix3m"] = np.nan
    if VVIX_PATH.exists():
        vvix = pd.read_csv(VVIX_PATH)
        vvix["datetime"] = pd.to_datetime(vvix["DATE"], errors="coerce")
        panel = panel.merge(vvix.rename(columns={"VVIX": "vvix"})[["datetime", "vvix"]], on="datetime", how="left")
    else:
        panel["vvix"] = np.nan
    panel["price"] = panel["panel_soxl_close"].astype(float)
    return panel.reset_index(drop=True)


def load_wf_actor(profile: str, window: str, position_col: str) -> pd.DataFrame:
    path = MODEL_CARD / profile / f"{profile}_{window}_soxl_daily.csv"
    if not path.exists():
        raise FileNotFoundError(path)
    df = pd.read_csv(path)
    df["datetime"] = pd.to_datetime(df["datetime"])
    if "position" not in df.columns:
        raise ValueError(f"missing position in {path}")
    return df[["datetime", "position"]].rename(columns={"position": position_col}).sort_values("datetime").reset_index(drop=True)


def compose_wf_base(window: str, market: pd.DataFrame) -> pd.DataFrame:
    slow = load_wf_actor(SLOW_PROFILE, window, "slow_position")
    fast = load_wf_actor(FAST_PROFILE, window, "fast_position")
    composed = compose_budget_usage_policy_frame(slow, fast, TwoSpeedBudgetConfig())
    composed["datetime"] = pd.to_datetime(composed["datetime"])
    price = market[["datetime", "price"]].copy()
    out = composed.merge(price, on="datetime", how="inner", validate="one_to_one")
    if len(out) != len(composed):
        raise ValueError(f"price alignment lost rows for {window}: {len(composed)} -> {len(out)}")
    return nav_from_positions(out, out["position"].to_numpy(dtype=float), f"risk_budget_usage_v1_{window}")


def nav_from_positions(frame: pd.DataFrame, positions: np.ndarray, label: str) -> pd.DataFrame:
    out = frame.copy().reset_index(drop=True)
    prices = out["price"].astype(float).to_numpy()
    positions = np.asarray(positions, dtype=float).clip(0.0, 1.0)
    nav = np.empty(len(out), dtype=float)
    nav[0] = 1_000_000.0
    for i in range(1, len(out)):
        nav[i] = nav[i - 1] * (1.0 + positions[i - 1] * (prices[i] / prices[i - 1] - 1.0))
    out["position"] = positions
    out["strategy_nav"] = nav
    out["buyhold_nav"] = 1_000_000.0 * prices / prices[0]
    out["excess"] = out["strategy_nav"] - out["buyhold_nav"]
    out["excess_pct"] = out["excess"] / out["buyhold_nav"] * 100.0
    delta = pd.Series(positions).diff().fillna(0.0)
    out["action"] = np.where(delta > 0.001, "BUY", np.where(delta < -0.001, "SELL", "HOLD"))
    out["source_label"] = label
    leading = ["datetime", "strategy_nav", "price", "buyhold_nav", "position", "excess", "excess_pct", "action", "source_label"]
    return out[leading + [c for c in out.columns if c not in leading]]


def max_drawdown_pct(nav: pd.Series) -> float:
    return float(-(nav / nav.cummax() - 1.0).min() * 100.0)


def max_drawdown_span(frame: pd.DataFrame) -> dict:
    z = frame.reset_index(drop=True)
    nav = z["strategy_nav"].astype(float)
    dd = nav / nav.cummax() - 1.0
    trough_idx = int(dd.idxmin())
    peak_idx = int(nav.loc[:trough_idx].idxmax())
    return {
        "peak_date": z.loc[peak_idx, "datetime"].strftime("%Y-%m-%d"),
        "trough_date": z.loc[trough_idx, "datetime"].strftime("%Y-%m-%d"),
        "max_dd": float(-dd.loc[trough_idx] * 100.0),
        "peak_position": float(z.loc[peak_idx, "position"]),
        "trough_position": float(z.loc[trough_idx, "position"]),
    }


def metrics(frame: pd.DataFrame, label: str, window: str, stage: str) -> dict:
    z = frame.sort_values("datetime").reset_index(drop=True)
    if len(z) < 2:
        raise ValueError(f"not enough rows for metrics: {label}")
    nav = z["strategy_nav"].astype(float)
    buyhold = z["buyhold_nav"].astype(float)
    daily = nav.pct_change().dropna()
    sharpe = 0.0 if daily.std(ddof=0) == 0 else float(daily.mean() / daily.std(ddof=0) * np.sqrt(252))
    ret = float((nav.iloc[-1] / nav.iloc[0] - 1.0) * 100.0)
    bh_ret = float((buyhold.iloc[-1] / buyhold.iloc[0] - 1.0) * 100.0)
    return {
        "window": window,
        "stage": stage,
        "label": label,
        "rows": int(len(z)),
        "start": z["datetime"].iloc[0].strftime("%Y-%m-%d"),
        "end": z["datetime"].iloc[-1].strftime("%Y-%m-%d"),
        "return": ret,
        "sharpe": sharpe,
        "max_dd": max_drawdown_pct(nav),
        "buy_and_hold_return": bh_ret,
        "relative_return": ret - bh_ret,
        "avg_pos": float(z["position"].astype(float).mean()),
        "trades": int((z["position"].astype(float).diff().abs() > 0.001).sum()),
        **{f"span_{k}": v for k, v in max_drawdown_span(z).items()},
    }


def saved_summary_compatible_metrics(frame: pd.DataFrame) -> dict:
    z = frame.sort_values("datetime").reset_index(drop=True)
    prices = z["price"].astype(float).to_numpy()
    positions = z["position"].astype(float).to_numpy()
    returns = np.diff(prices) / prices[:-1]
    strategy_returns = positions[:-1] * returns
    nav = np.cumprod(1.0 + strategy_returns)
    buyhold = np.cumprod(1.0 + returns)
    peak = np.maximum.accumulate(nav)
    drawdown = (nav - peak) / peak
    return {
        "return": float(nav[-1] - 1.0) * 100.0,
        "max_dd": abs(float(np.min(drawdown))) * 100.0,
        "buy_and_hold_return": float(buyhold[-1] - 1.0) * 100.0,
        "relative_return": float(nav[-1] - buyhold[-1]) * 100.0,
        "avg_pos": float(np.mean(positions)),
    }


def rolling_percentile_last(x: np.ndarray) -> float:
    x = x[~np.isnan(x)]
    return np.nan if len(x) == 0 else float((x <= x[-1]).mean())


def build_feature_frame(market: pd.DataFrame) -> pd.DataFrame:
    f = market[["datetime", "price", "panel_soxl_close", "panel_soxx_close", "vix", "vix3m", "vvix", "soxl_volume", "soxx_volume"]].copy()
    price = f["price"].astype(float)
    soxx = f["panel_soxx_close"].astype(float)
    soxl_ret = price.pct_change()
    soxx_ret = soxx.pct_change()
    log_ret = np.log(price).diff()
    for w in [5, 10, 21, 63, 126]:
        f[f"mom_{w}"] = price / price.shift(w) - 1.0
        f[f"drawdown_{w}"] = price / price.rolling(w, min_periods=max(5, w // 3)).max() - 1.0
        f[f"rv_{w}"] = log_ret.rolling(w, min_periods=max(5, w // 3)).std() * np.sqrt(252)
        f[f"downside_vol_{w}"] = log_ret.clip(upper=0).rolling(w, min_periods=max(5, w // 3)).std() * np.sqrt(252)
    for w in [50, 100, 200]:
        ma = price.rolling(w, min_periods=max(20, w // 2)).mean()
        f[f"ma{w}_gap"] = price / ma - 1.0
    f["days_below_ma100_63"] = (f["ma100_gap"] < 0).rolling(63, min_periods=21).mean()
    f["days_below_ma200_63"] = (f["ma200_gap"] < 0).rolling(63, min_periods=21).mean()
    f["vol_of_vol_21"] = f["rv_10"].rolling(21, min_periods=10).std()
    f["vol_accel_10_21"] = f["rv_10"] - f["rv_21"]
    ratio = price / soxx
    f["soxl_soxx_ratio_widen_5"] = ratio / ratio.shift(5) - 1.0
    f["soxl_soxx_ratio_widen_21"] = ratio / ratio.shift(21) - 1.0
    f["soxl_soxx_ratio_z_252"] = (ratio - ratio.rolling(252, min_periods=63).mean()) / ratio.rolling(252, min_periods=63).std()
    f["signed_te_reversion_10"] = (soxl_ret - 3.0 * soxx_ret).rolling(10, min_periods=5).sum()
    f["soxx_mom_21"] = soxx / soxx.shift(21) - 1.0
    f["soxx_bias_10"] = soxx / soxx.rolling(10, min_periods=5).mean() - 1.0
    f["rel_volume_ratio_21"] = (f["soxl_volume"] / f["soxl_volume"].rolling(21, min_periods=10).mean()) - (f["soxx_volume"] / f["soxx_volume"].rolling(21, min_periods=10).mean())
    f["vix_pct_252"] = f["vix"].rolling(252, min_periods=63).apply(rolling_percentile_last, raw=True)
    f["vix_z_126"] = (f["vix"] - f["vix"].rolling(126, min_periods=42).mean()) / f["vix"].rolling(126, min_periods=42).std()
    f["vix_chg_5"] = f["vix"] / f["vix"].shift(5) - 1.0
    f["vix_term"] = f["vix"] / f["vix3m"] - 1.0
    f["vvix_z_126"] = (f["vvix"] - f["vvix"].rolling(126, min_periods=42).mean()) / f["vvix"].rolling(126, min_periods=42).std()
    for col in [c for c in f.columns if c not in {"datetime", "price"}]:
        f[f"lag_{col}"] = f[col].shift(1)
    return f


def percentile_against_calibration(values: pd.Series, calib_mask: pd.Series) -> pd.Series:
    arr = values.astype(float).to_numpy()
    calib = values[calib_mask].dropna().astype(float).to_numpy()
    out = np.full(len(values), np.nan, dtype=float)
    if len(calib) < 30:
        return pd.Series(out, index=values.index)
    calib = np.sort(calib)
    valid = ~np.isnan(arr)
    out[valid] = np.searchsorted(calib, arr[valid], side="right") / len(calib)
    return pd.Series(out, index=values.index).clip(0.0, 1.0)


def build_score_frame(market: pd.DataFrame) -> tuple[pd.DataFrame, dict[str, list[str]]]:
    score_specs = {
        "panic_rebound_guarded": {
            "positive_risk": ["lag_vix_chg_5", "lag_vix_z_126", "lag_downside_vol_10", "lag_vol_accel_10_21", "lag_days_below_ma200_63"],
            "negative_risk": ["lag_mom_21", "lag_ma50_gap", "lag_soxl_soxx_ratio_widen_5"],
        }
    }
    features = build_feature_frame(market)
    calib_mask = features["datetime"] <= pd.Timestamp("2023-12-31")
    out = features.copy()
    components: dict[str, list[str]] = {}
    for score_name, spec in score_specs.items():
        comps = []
        for col in spec["positive_risk"]:
            if col in out.columns:
                comp = f"riskpct_{score_name}_{col}"
                out[comp] = percentile_against_calibration(out[col], calib_mask)
                comps.append(comp)
        for col in spec["negative_risk"]:
            if col in out.columns:
                comp = f"riskpct_{score_name}_neg_{col}"
                out[comp] = percentile_against_calibration(-out[col], calib_mask)
                comps.append(comp)
        components[score_name] = comps
        out[f"score_{score_name}"] = out[comps].mean(axis=1, skipna=True)
    keep = ["datetime", "score_panic_rebound_guarded", "lag_mom_21", "lag_ma100_gap", "lag_ma50_gap", "lag_vix_pct_252", "lag_downside_vol_10", "lag_vol_accel_10_21"]
    return out[[c for c in keep if c in out.columns]].copy(), components


def build_downside_scale_frame(market: pd.DataFrame, config: dict) -> pd.DataFrame:
    price = market["price"].astype(float)
    returns = price.pct_change()
    downside_risk = returns.clip(upper=0).rolling(config["window"], min_periods=max(5, config["window"] // 2)).std() * np.sqrt(252)
    scale = (config["target_vol"] / downside_risk).replace([np.inf, -np.inf], np.nan).clip(0.0, config["max_scale"]).fillna(1.0)
    if config["trend_filter"]:
        ma200 = price.rolling(200, min_periods=50).mean()
        mom21 = price / price.shift(21) - 1.0
        scale = pd.Series(
            np.where((price > ma200) & (mom21 > 0.0), scale, np.minimum(scale, 1.0)),
            index=market.index,
        ).fillna(1.0)
    return pd.DataFrame({"datetime": market["datetime"], "downside_risk_estimate": downside_risk, "vol_scale": scale})


def apply_downside_vol_target(frame: pd.DataFrame, scale_frame: pd.DataFrame, config: dict) -> pd.DataFrame:
    out = frame.merge(scale_frame, on="datetime", how="left", validate="one_to_one")
    if out["vol_scale"].isna().any():
        raise ValueError("missing volatility scale rows")
    base_pos = out["position"].astype(float).to_numpy()
    desired = np.clip(base_pos * out["vol_scale"].astype(float).to_numpy(), 0.0, 1.0)
    positions: list[float] = []
    current = 0.0
    for target in desired:
        target = float(target)
        if abs(target - current) >= config["band"]:
            current = target
        positions.append(current)
    scored = nav_from_positions(out, np.array(positions, dtype=float), config["label"])
    scored["downside_base_position"] = base_pos
    scored["downside_changed"] = np.abs(scored["position"].to_numpy(dtype=float) - base_pos) > 1e-9
    return scored


@dataclass(frozen=True)
class OverlayConfig:
    score_name: str
    entry: float
    exit: float
    cap: float
    min_account_dd: float
    trade_band: float
    release_mom: bool


def apply_score_overlay(frame: pd.DataFrame, config: OverlayConfig) -> tuple[pd.DataFrame, dict]:
    source = frame.reset_index(drop=True).copy()
    score_cols = ["score_panic_rebound_guarded", "lag_mom_21", "lag_ma100_gap", "lag_ma50_gap", "lag_vix_pct_252", "lag_downside_vol_10", "lag_vol_accel_10_21"]
    source = source.drop(columns=[c for c in score_cols if c in source.columns], errors="ignore")
    source = source.merge(SCORE_FRAME, on="datetime", how="left", validate="one_to_one")
    base_pos = source["position"].astype(float).to_numpy()
    prices = source["price"].astype(float).to_numpy()
    score = source[f"score_{config.score_name}"].astype(float).to_numpy()
    mom21 = source.get("lag_mom_21", pd.Series(np.nan, index=source.index)).astype(float).to_numpy()
    ma100 = source.get("lag_ma100_gap", pd.Series(np.nan, index=source.index)).astype(float).to_numpy()
    positions = np.empty(len(source), dtype=float)
    risk_state = np.zeros(len(source), dtype=bool)
    account_dd = np.zeros(len(source), dtype=float)
    nav = 1.0
    peak = 1.0
    current = float(base_pos[0])
    state = False
    risk_active_days = 0
    cap_changed_days = 0
    for i in range(len(source)):
        if i > 0:
            nav *= 1.0 + current * (prices[i] / prices[i - 1] - 1.0)
            peak = max(peak, nav)
        dd = 1.0 - nav / peak if peak else 0.0
        account_dd[i] = dd
        s = score[i]
        if np.isnan(s):
            s = 0.0
        release_ok = config.release_mom and (not np.isnan(mom21[i])) and (not np.isnan(ma100[i])) and mom21[i] > 0 and ma100[i] > 0
        if state:
            if s <= config.exit or release_ok:
                state = False
        elif s >= config.entry and dd >= config.min_account_dd:
            state = True
        desired = min(base_pos[i], config.cap) if state else base_pos[i]
        if abs(desired - current) < config.trade_band:
            desired = current
        if state:
            risk_active_days += 1
        if desired < base_pos[i] - 1e-9:
            cap_changed_days += 1
        current = float(np.clip(desired, 0.0, 1.0))
        positions[i] = current
        risk_state[i] = state
    label = f"{config.score_name}_e{config.entry:.2f}_x{config.exit:.2f}_cap{config.cap:.2f}_dd{config.min_account_dd:.2f}_band{config.trade_band:.2f}_rel{int(config.release_mom)}"
    out = nav_from_positions(source, positions, label)
    out["crash_base_position"] = base_pos
    out["crash_risk_score"] = source[f"score_{config.score_name}"].to_numpy()
    out["crash_risk_state"] = risk_state
    out["crash_risk_account_drawdown"] = account_dd
    out["crash_risk_changed"] = out["position"] < out["crash_base_position"] - 1e-9
    return out, {
        "label": label,
        "score_name": config.score_name,
        "entry": config.entry,
        "exit": config.exit,
        "cap": config.cap,
        "min_account_dd": config.min_account_dd,
        "trade_band": config.trade_band,
        "release_mom": config.release_mom,
        "risk_active_days": int(risk_active_days),
        "cap_changed_days": int(cap_changed_days),
    }


@dataclass(frozen=True)
class ResidualGuardConfig:
    start_drawdown: float
    full_drawdown: float
    min_cap: float
    trade_band: float
    release_drawdown: float


def apply_residual_guard(frame: pd.DataFrame, config: ResidualGuardConfig) -> tuple[pd.DataFrame, dict]:
    source = frame.reset_index(drop=True).copy()
    base_pos = source["position"].astype(float).to_numpy()
    prices = source["price"].astype(float).to_numpy()
    positions = np.empty(len(source), dtype=float)
    account_dd = np.zeros(len(source), dtype=float)
    guard_cap = np.ones(len(source), dtype=float)
    guard_active = np.zeros(len(source), dtype=bool)
    nav = 1.0
    peak = 1.0
    current = float(base_pos[0])
    active_days = 0
    changed_days = 0
    for i in range(len(source)):
        if i > 0:
            nav *= 1.0 + current * (prices[i] / prices[i - 1] - 1.0)
            peak = max(peak, nav)
        dd = 1.0 - nav / peak if peak else 0.0
        account_dd[i] = dd
        if dd <= config.release_drawdown:
            desired = base_pos[i]
            cap = 1.0
        elif dd >= config.start_drawdown:
            scale = 1.0 - min(1.0, max(0.0, (dd - config.start_drawdown) / (config.full_drawdown - config.start_drawdown)))
            dynamic_cap = max(config.min_cap, base_pos[i] * scale)
            desired = min(base_pos[i], dynamic_cap)
            cap = dynamic_cap
            active_days += 1
        else:
            desired = base_pos[i]
            cap = 1.0
        if abs(desired - current) < config.trade_band:
            desired = current
        if desired < base_pos[i] - 1e-9:
            changed_days += 1
        current = float(np.clip(desired, 0.0, 1.0))
        positions[i] = current
        guard_cap[i] = cap
        guard_active[i] = dd >= config.start_drawdown
    label = f"resid_start{config.start_drawdown:.3f}_full{config.full_drawdown:.3f}_cap{config.min_cap:.3f}_band{config.trade_band:.2f}_rel{config.release_drawdown:.2f}"
    out = nav_from_positions(source, positions, label)
    out["residual_guard_base_position"] = base_pos
    out["residual_guard_account_drawdown"] = account_dd
    out["residual_guard_dynamic_cap"] = guard_cap
    out["residual_guard_active"] = guard_active
    out["residual_guard_changed"] = out["position"] < out["residual_guard_base_position"] - 1e-9
    return out, {
        "label": label,
        "start_drawdown": config.start_drawdown,
        "full_drawdown": config.full_drawdown,
        "min_cap": config.min_cap,
        "trade_band": config.trade_band,
        "release_drawdown": config.release_drawdown,
        "active_days": int(active_days),
        "changed_days": int(changed_days),
    }


def write_daily(frame: pd.DataFrame, window: str, stage: str) -> str:
    path = WALK_DIR / f"{EXPERIMENT}_{window}_{stage}_soxl_daily.csv"
    frame.to_csv(path, index=False)
    return str(path.relative_to(ROOT))


market = load_market_panel()
SCORE_FRAME, SCORE_COMPONENTS = build_score_frame(market)
VOL_SCALE_FRAME = build_downside_scale_frame(market, DOWNSIDE_CONFIG)
base_wf_summary = json.loads(BASE_WF_SUMMARY_PATH.read_text())
previous_selection = json.loads(PREVIOUS_SELECTION_PATH.read_text())
crash_cfg = OverlayConfig(**CRASH_CONFIG)
resid_cfg = ResidualGuardConfig(**RESIDUAL_CONFIG)

rows = []
artifacts: dict[str, dict[str, str]] = {}
meta_rows = []
frames_by_window: dict[str, dict[str, pd.DataFrame]] = {}
for window in WF_WINDOWS:
    base = compose_wf_base(window, market)
    downside = apply_downside_vol_target(base, VOL_SCALE_FRAME, DOWNSIDE_CONFIG)
    crash, crash_meta = apply_score_overlay(downside, crash_cfg)
    residual, residual_meta = apply_residual_guard(crash, resid_cfg)
    frames = {"base": base, "downside_vol": downside, "crash_risk": crash, "residual_guard": residual}
    frames_by_window[window] = frames
    artifacts[window] = {stage: write_daily(frame, window, stage) for stage, frame in frames.items()}
    for stage, frame in frames.items():
        rows.append(metrics(frame, frame["source_label"].iloc[-1], window, stage))
    meta_rows.append({"window": window, "overlay": "crash_risk", **crash_meta})
    meta_rows.append({"window": window, "overlay": "residual_guard", **residual_meta})

results = pd.DataFrame(rows)
summary_check_rows = []
for window in WF_WINDOWS:
    saved = base_wf_summary["windows"][window]
    observed_nav = results[(results["window"] == window) & (results["stage"] == "base")].iloc[0]
    observed_saved = saved_summary_compatible_metrics(frames_by_window[window]["base"])
    summary_check_rows.append({
        "window": window,
        "delta_return": float(observed_saved["return"] - saved["return"]),
        "delta_policy_max_dd": float(observed_saved["max_dd"] - saved["max_dd"]),
        "delta_relative_return": float(observed_saved["relative_return"] - saved["relative_return"]),
        "delta_avg_pos": float(observed_saved["avg_pos"] - saved["avg_pos"]),
        "nav_metric_max_dd": float(observed_nav["max_dd"]),
        "saved_metric_max_dd": float(saved["max_dd"]),
        "max_dd_convention_delta": float(observed_nav["max_dd"] - observed_saved["max_dd"]),
    })
summary_check = pd.DataFrame(summary_check_rows)

stage_pivot = results.pivot(index="window", columns="stage", values=["return", "max_dd", "relative_return", "avg_pos", "trades"]) 
comparison_rows = []
for window in WF_WINDOWS:
    crash = results[(results["window"] == window) & (results["stage"] == "crash_risk")].iloc[0]
    residual = results[(results["window"] == window) & (results["stage"] == "residual_guard")].iloc[0]
    base = results[(results["window"] == window) & (results["stage"] == "base")].iloc[0]
    comparison_rows.append({
        "window": window,
        "residual_vs_crash_delta_return": float(residual["return"] - crash["return"]),
        "residual_vs_crash_delta_max_dd": float(residual["max_dd"] - crash["max_dd"]),
        "residual_vs_crash_delta_relative_return": float(residual["relative_return"] - crash["relative_return"]),
        "residual_target_pass": bool(residual["relative_return"] >= 30.0 and residual["max_dd"] < 15.0),
        "residual_risk_pass": bool(residual["max_dd"] < 15.0),
        "residual_mechanism_pass": bool(residual["max_dd"] <= crash["max_dd"] + 1e-9 and residual["relative_return"] >= crash["relative_return"] - 5.0),
        "base_max_dd": float(base["max_dd"]),
        "crash_max_dd": float(crash["max_dd"]),
        "residual_max_dd": float(residual["max_dd"]),
        "base_relative_return": float(base["relative_return"]),
        "crash_relative_return": float(crash["relative_return"]),
        "residual_relative_return": float(residual["relative_return"]),
    })
comparison = pd.DataFrame(comparison_rows)
meta = pd.DataFrame(meta_rows)

wf_risk_pass = bool(comparison["residual_risk_pass"].all())
wf_mechanism_pass = bool(comparison["residual_mechanism_pass"].all())
wf_strict_target_windows = int(comparison["residual_target_pass"].sum())
base_summary_match = bool((summary_check[["delta_return", "delta_policy_max_dd", "delta_relative_return", "delta_avg_pos"]].abs() < 1e-6).all().all())
if wf_risk_pass and wf_mechanism_pass and base_summary_match:
    decision = "walkforward_mechanism_pass_needs_runtime_profile"
elif wf_mechanism_pass and base_summary_match:
    decision = "walkforward_mechanism_pass_risk_target_failed"
elif base_summary_match:
    decision = "candidate_blocked_by_walkforward"
else:
    decision = "invalid_reconstruction_check_failed"

results_path = CARD / "walkforward_stage_metrics.csv"
comparison_path = CARD / "walkforward_comparison.csv"
summary_check_path = CARD / "base_reconstruction_check.csv"
meta_path = CARD / "overlay_meta.csv"
selection_path = CARD / "selection.json"
results.to_csv(results_path, index=False)
comparison.to_csv(comparison_path, index=False)
summary_check.to_csv(summary_check_path, index=False)
meta.to_csv(meta_path, index=False)

selection = {
    "experiment": EXPERIMENT,
    "baseline_walkforward_summary_path": str(BASE_WF_SUMMARY_PATH.relative_to(ROOT)),
    "previous_candidate_selection_path": str(PREVIOUS_SELECTION_PATH.relative_to(ROOT)),
    "previous_candidate_decision": previous_selection.get("decision"),
    "slow_profile": SLOW_PROFILE,
    "fast_profile": FAST_PROFILE,
    "windows": WF_DATE_RANGES,
    "boundary": "Reconstructs selected downside-vol, crash-risk, and residual guard overlays on saved risk_budget_usage_v1 slow/fast WF daily actor paths; account drawdown state resets by WF window.",
    "configs": {
        "downside_vol": DOWNSIDE_CONFIG,
        "crash_risk": CRASH_CONFIG,
        "residual_guard": RESIDUAL_CONFIG,
    },
    "score_components": SCORE_COMPONENTS,
    "base_summary_match": base_summary_match,
    "wf_risk_pass": wf_risk_pass,
    "wf_mechanism_pass": wf_mechanism_pass,
    "strict_target_windows": wf_strict_target_windows,
    "decision": decision,
    "results": results.to_dict(orient="records"),
    "comparison": comparison.to_dict(orient="records"),
    "artifacts": {
        "notebook": str((CARD / f"{EXPERIMENT}.ipynb").relative_to(ROOT)),
        "stage_metrics": str(results_path.relative_to(ROOT)),
        "comparison": str(comparison_path.relative_to(ROOT)),
        "base_reconstruction_check": str(summary_check_path.relative_to(ROOT)),
        "overlay_meta": str(meta_path.relative_to(ROOT)),
        "walkforward_daily": artifacts,
        "selection": str(selection_path.relative_to(ROOT)),
    },
}
selection_path.write_text(json.dumps(selection, indent=2, ensure_ascii=False) + "\n")

readout = [
    "## Final Readout",
    f"Decision: `{decision}`.",
    f"Base reconstruction check: `{base_summary_match}` (saved-summary-compatible metric deltas < 1e-6; NAV MaxDD convention delta is reported separately).",
    f"WF risk pass (`residual max_dd < 15%` in every window): `{wf_risk_pass}`.",
    f"WF mechanism pass (`max_dd` not worse and relative return no worse than crash-risk by 5pp): `{wf_mechanism_pass}`.",
    f"Strict target windows (`relative >= 30pp` and `max_dd < 15%`): `{wf_strict_target_windows}` / `{len(WF_WINDOWS)}`.",
    "Conclusion: promotion is allowed only if reconstruction, risk, and mechanism gates all pass. Otherwise keep this as runtime evidence and do not promote the residual guard candidate.",
    "Saved artifacts:",
    f"- `{selection['artifacts']['stage_metrics']}`",
    f"- `{selection['artifacts']['comparison']}`",
    f"- `{selection['artifacts']['base_reconstruction_check']}`",
    f"- `{selection['artifacts']['overlay_meta']}`",
    f"- `{selection['artifacts']['selection']}`",
]

display(Markdown("\n\n".join(readout)))
display(Markdown("## Stage Metrics"))
display(results[["window", "stage", "return", "sharpe", "max_dd", "relative_return", "avg_pos", "trades", "span_peak_date", "span_trough_date"]])
display(Markdown("## Residual vs Crash-Risk Comparison"))
display(comparison)
display(Markdown("## Base Reconstruction Check"))
display(summary_check)
display(Markdown("## Overlay Meta"))
display(meta)
print(json.dumps(selection, indent=2, ensure_ascii=False))

## Final Readout

Decision: `candidate_blocked_by_walkforward`.

Base reconstruction check: `True` (saved-summary-compatible metric deltas < 1e-6; NAV MaxDD convention delta is reported separately).

WF risk pass (`residual max_dd < 15%` in every window): `False`.

WF mechanism pass (`max_dd` not worse and relative return no worse than crash-risk by 5pp): `False`.

Strict target windows (`relative >= 30pp` and `max_dd < 15%`): `1` / `3`.

Conclusion: promotion is allowed only if reconstruction, risk, and mechanism gates all pass. Otherwise keep this as runtime evidence and do not promote the residual guard candidate.

Saved artifacts:

- `runtime/execution/card/risk_budget_residual_drawdown_guard_walkforward_v1/walkforward_stage_metrics.csv`

- `runtime/execution/card/risk_budget_residual_drawdown_guard_walkforward_v1/walkforward_comparison.csv`

- `runtime/execution/card/risk_budget_residual_drawdown_guard_walkforward_v1/base_reconstruction_check.csv`

- `runtime/execution/card/risk_budget_residual_drawdown_guard_walkforward_v1/overlay_meta.csv`

- `runtime/execution/card/risk_budget_residual_drawdown_guard_walkforward_v1/selection.json`

## Stage Metrics

,window,stage,return,sharpe,max_dd,relative_return,avg_pos,trades,span_peak_date,span_trough_date
0,wf_2020,base,58.616237,1.130603,28.569218,-1.159072,0.316998,18,2020-02-19,2020-03-20
1,wf_2020,downside_vol,25.501711,0.830902,19.709418,-34.273597,0.229659,57,2020-09-02,2020-09-23
2,wf_2020,crash_risk,21.417928,0.739137,22.123347,-38.357381,0.221486,48,2020-09-02,2020-09-23
3,wf_2020,residual_guard,5.580720,0.338053,18.102212,-54.194588,0.172200,69,2020-09-02,2020-09-23
4,wf_2022,base,-18.643970,-1.582119,20.244361,67.944102,0.078045,11,2022-01-03,2022-09-30
5,wf_2022,downside_vol,-11.817901,-1.847116,13.201112,74.770171,0.041406,25,2022-01-03,2022-09-30
6,wf_2022,crash_risk,-9.930770,-1.463625,10.970665,76.657302,0.045817,8,2022-01-03,2022-09-30
7,wf_2022,residual_guard,-9.930770,-1.463625,10.970665,76.657302,0.045817,8,2022-01-03,2022-09-30
8,wf_2023_2024,base,28.805671,0.983661,27.165913,31.409100,0.214009,25,2024-03-07,2024-04-19
9,wf_2023_2024,downside_vol,41.477093,1.250937,22.294298,44.080522,0.192546,56,2024-03-07,2024-04-19


## Residual vs Crash-Risk Comparison

,window,residual_vs_crash_delta_return,residual_vs_crash_delta_max_dd,residual_vs_crash_delta_relative_return,residual_target_pass,residual_risk_pass,residual_mechanism_pass,base_max_dd,crash_max_dd,residual_max_dd,base_relative_return,crash_relative_return,residual_relative_return
0,wf_2020,-15.837208,-4.021135,-15.837208,False,False,False,28.569218,22.123347,18.102212,-1.159072,-38.357381,-54.194588
1,wf_2022,0.000000,0.000000,0.000000,True,True,True,20.244361,10.970665,10.970665,67.944102,76.657302,76.657302
2,wf_2023_2024,-4.726557,-1.849897,-4.726557,False,False,True,27.165913,19.079874,17.229977,31.409100,43.037195,38.310638


## Base Reconstruction Check

,window,delta_return,delta_policy_max_dd,delta_relative_return,delta_avg_pos,nav_metric_max_dd,saved_metric_max_dd,max_dd_convention_delta
0,wf_2020,4.263256e-14,0.000000e+00,2.131628e-14,0.0,28.569218,28.569218,-1.065814e-14
1,wf_2022,2.131628e-14,-2.131628e-14,4.263256e-14,0.0,20.244361,20.144541,9.981921e-02
2,wf_2023_2024,0.000000e+00,0.000000e+00,0.000000e+00,0.0,27.165913,27.165913,0.000000e+00


## Overlay Meta

,window,overlay,label,score_name,entry,exit,cap,min_account_dd,trade_band,release_mom,risk_active_days,cap_changed_days,start_drawdown,full_drawdown,min_cap,release_drawdown,active_days,changed_days
0,wf_2020,crash_risk,panic_rebound_guarded_e0.70_x0.65_cap0.05_dd0....,panic_rebound_guarded,0.7,0.65,0.05,0.07,0.04,False,22.0,50.0,NaN,NaN,NaN,NaN,NaN,NaN
1,wf_2020,residual_guard,resid_start0.090_full0.220_cap0.095_band0.01_r...,NaN,NaN,NaN,NaN,NaN,0.01,NaN,NaN,NaN,0.09,0.22,0.095,0.0,137.0,69.0
2,wf_2022,crash_risk,panic_rebound_guarded_e0.70_x0.65_cap0.05_dd0....,panic_rebound_guarded,0.7,0.65,0.05,0.07,0.04,False,65.0,43.0,NaN,NaN,NaN,NaN,NaN,NaN
3,wf_2022,residual_guard,resid_start0.090_full0.220_cap0.095_band0.01_r...,NaN,NaN,NaN,NaN,NaN,0.01,NaN,NaN,NaN,0.09,0.22,0.095,0.0,86.0,0.0
4,wf_2023_2024,crash_risk,panic_rebound_guarded_e0.70_x0.65_cap0.05_dd0....,panic_rebound_guarded,0.7,0.65,0.05,0.07,0.04,False,60.0,45.0,NaN,NaN,NaN,NaN,NaN,NaN
5,wf_2023_2024,residual_guard,resid_start0.090_full0.220_cap0.095_band0.01_r...,NaN,NaN,NaN,NaN,NaN,0.01,NaN,NaN,NaN,0.09,0.22,0.095,0.0,211.0,75.0


{
  "experiment": "risk_budget_residual_drawdown_guard_walkforward_v1",
  "baseline_walkforward_summary_path": "runtime/model/card/risk_budget_usage_v1/risk_budget_usage_v1_walkforward_composed.json",
  "previous_candidate_selection_path": "runtime/execution/card/risk_budget_residual_drawdown_guard_v1/selection.json",
  "previous_candidate_decision": "candidate_target_pass_needs_walkforward",
  "slow_profile": "balanced_rule_residual_mode_v1",
  "fast_profile": "balanced_rule_residual_imitation_v4",
  "windows": {
    "wf_2020": [
      "2020-01-01",
      "2020-12-31"
    ],
    "wf_2022": [
      "2022-01-01",
      "2022-12-31"
    ],
    "wf_2023_2024": [
      "2023-01-01",
      "2024-12-31"
    ]
  },
  "boundary": "Reconstructs selected downside-vol, crash-risk, and residual guard overlays on saved risk_budget_usage_v1 slow/fast WF daily actor paths; account drawdown state resets by WF window.",
  "configs": {
    "downside_vol": {
      "label": "vol_w10_tv0.35_mx4.0_b0.02_tre